# 🛍️ Silver Trust — AI MVP
**Streetwear E-Commerce · Germany & Europe**

This notebook demonstrates three things end-to-end:
1. **Mini storefront** — two streetwear articles with a product catalog
2. **Recommendation engine** — embedding-based similarity ranking
3. **AI Styling Assistant** — RAG-powered conversational advisor

Every LLM call is traced in **LangSmith** for full observability.

---
> **Before running:** fill in your keys in Cell 2.

## 0 · Dependencies

In [ ]:
# Install / upgrade required packages (run once)
import subprocess, sys
pkgs = [
    "openai",
    "langchain",
    "langchain-openai",
    "langsmith",
    "python-dotenv",
    "numpy",
    "IPython",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade"] + pkgs)
print("✅ All packages ready")

## 1 · Configuration

Keys are loaded from a `.env` file in the same directory as this notebook.  
**Never commit `.env` to version control** — add it to `.gitignore`.

```
# Copy .env.example → .env, then fill in your values
cp .env.example .env
```

In [ ]:
import os
from dotenv import load_dotenv

# ── Load keys from .env (never commit that file) ─────────────────────────────
load_dotenv()   # reads .env from the same directory as this notebook

# Verify required keys are present
_required = ["OPENAI_API_KEY", "LANGSMITH_API_KEY"]
_missing  = [k for k in _required if not os.getenv(k)]
if _missing:
    raise EnvironmentError(
        f"Missing keys in .env: {', '.join(_missing)}\n"
        "Copy .env.example → .env and fill in your values."
    )

# ── LangSmith settings (non-secret — safe to keep here) ──────────────────────
os.environ.setdefault("LANGSMITH_TRACING",  "true")
os.environ.setdefault("LANGSMITH_PROJECT",  "silver-trust-mvp")
os.environ.setdefault("LANGSMITH_ENDPOINT", "https://eu.api.smith.langchain.com")

# ── Model ─────────────────────────────────────────────────────────────────────
CHAT_MODEL      = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

print("✅ Config loaded")
print(f"   LangSmith project : {os.environ['LANGSMITH_PROJECT']}")
print(f"   LangSmith endpoint: {os.environ['LANGSMITH_ENDPOINT']}")
print(f"   Chat model        : {CHAT_MODEL}")
print(f"   Embedding model   : {EMBEDDING_MODEL}")

## 2 · Product Catalog

Two hero articles for the MVP, plus six supporting items used by the recommendation engine.

In [ ]:
from dataclasses import dataclass, field
from typing import List, Optional

@dataclass
class Product:
    id: str
    name: str
    category: str
    price: float
    colours: List[str]
    sizes: List[str]
    description: str
    tags: List[str]
    embedding: Optional[List[float]] = field(default=None, repr=False)

    def catalog_text(self) -> str:
        """Text representation injected into the LLM context (RAG)."""
        return (
            f"[{self.id}] {self.name} | {self.category} | €{self.price:.2f} | "
            f"Colours: {', '.join(self.colours)} | Sizes: {', '.join(self.sizes)} | "
            f"{self.description} | Tags: {', '.join(self.tags)}"
        )


# ── The two hero articles shown on the storefront ───────────────────────────
HERO_PRODUCTS = [
    Product(
        id="ST-001",
        name="Köln Oversized Hoodie",
        category="Tops",
        price=89.90,
        colours=["Washed Black", "Off White", "Sage Green"],
        sizes=["XS", "S", "M", "L", "XL", "XXL"],
        description=(
            "400 gsm French terry, dropped shoulders, kangaroo pocket. "
            "Heavyweight oversized fit — wears like a statement, feels like a blanket."
        ),
        tags=["hoodie", "oversized", "heavyweight", "unisex", "bestseller"],
    ),
    Product(
        id="ST-002",
        name="Dortmund Cargo Pants",
        category="Bottoms",
        price=119.90,
        colours=["Olive", "Concrete Grey", "Off Black"],
        sizes=["28", "30", "32", "34", "36"],
        description=(
            "Relaxed-fit cargo with six functional pockets, adjustable ankle cuffs, "
            "and a nylon ripstop shell. Built for the city, ready for anything."
        ),
        tags=["cargo", "relaxed", "pockets", "utility", "unisex"],
    ),
]

# ── Supporting catalog — feeds the recommendation engine ────────────────────
SUPPORT_PRODUCTS = [
    Product(
        id="ST-003",
        name="Essen Slim Jogger",
        category="Bottoms",
        price=79.90,
        colours=["Black", "Charcoal"],
        sizes=["XS", "S", "M", "L", "XL"],
        description="Tapered slim jogger in brushed cotton. Ribbed cuffs, zip pocket.",
        tags=["jogger", "slim", "tapered", "cotton", "everyday"],
    ),
    Product(
        id="ST-004",
        name="Düsseldorf Graphic Tee",
        category="Tops",
        price=49.90,
        colours=["White", "Black", "Rust"],
        sizes=["XS", "S", "M", "L", "XL", "XXL"],
        description="220 gsm heavyweight tee. Screen-printed chest graphic. Boxy fit.",
        tags=["tee", "graphic", "boxy", "cotton", "streetwear"],
    ),
    Product(
        id="ST-005",
        name="Hamburg Track Jacket",
        category="Outerwear",
        price=139.90,
        colours=["Navy", "Black", "Burgundy"],
        sizes=["S", "M", "L", "XL"],
        description="Retro track jacket, contrast piping, full-zip, two side pockets.",
        tags=["jacket", "track", "retro", "zip", "outerwear"],
    ),
    Product(
        id="ST-006",
        name="Berlin Beanie",
        category="Accessories",
        price=29.90,
        colours=["Black", "Olive", "Grey"],
        sizes=["One Size"],
        description="Ribbed merino-blend beanie. Embroidered logo tab. Foldable cuff.",
        tags=["beanie", "hat", "accessories", "winter", "merino"],
    ),
    Product(
        id="ST-007",
        name="Frankfurt Crossbody Bag",
        category="Accessories",
        price=69.90,
        colours=["Black", "Olive"],
        sizes=["One Size"],
        description="600D nylon crossbody. Adjustable strap, external zip pocket, 4L capacity.",
        tags=["bag", "crossbody", "accessories", "nylon", "utility"],
    ),
    Product(
        id="ST-008",
        name="München Crewneck Sweatshirt",
        category="Tops",
        price=99.90,
        colours=["Stone", "Black", "Forest Green"],
        sizes=["XS", "S", "M", "L", "XL", "XXL"],
        description="360 gsm fleece-back crew. Relaxed fit, ribbed cuffs and hem.",
        tags=["crewneck", "sweatshirt", "relaxed", "heavyweight", "fleece"],
    ),
]

ALL_PRODUCTS = HERO_PRODUCTS + SUPPORT_PRODUCTS
CATALOG_INDEX = {p.id: p for p in ALL_PRODUCTS}

print(f"✅ Catalog loaded — {len(ALL_PRODUCTS)} products ({len(HERO_PRODUCTS)} hero, {len(SUPPORT_PRODUCTS)} supporting)")

## 3 · Storefront Display

A clean HTML product card for each hero article, rendered inline in the notebook.

In [ ]:
from IPython.display import display, HTML

PRODUCT_IMAGES = {
    "ST-001": "https://images.unsplash.com/photo-1542291026-7eec264c27ff?w=400&q=80",  # hoodie
    "ST-002": "https://images.unsplash.com/photo-1624378439575-d8705ad7ae80?w=400&q=80",  # cargo
}

def render_storefront(products):
    cards = ""
    for p in products:
        img_url = PRODUCT_IMAGES.get(p.id, "")
        colour_dots = " ".join(
            f'<span title="{c}" style="display:inline-block;width:14px;height:14px;'
            f'border-radius:50%;background:{c.lower().replace(" ","")};border:1px solid #ccc;margin-right:4px;"></span>'
            for c in p.colours
        )
        size_pills = " ".join(
            f'<span style="border:1px solid #333;padding:2px 8px;font-size:11px;border-radius:3px;">{s}</span>'
            for s in p.sizes
        )
        tags_html = " ".join(
            f'<span style="background:#f0f0f0;padding:2px 7px;font-size:10px;border-radius:10px;color:#555;">{t}</span>'
            for t in p.tags
        )
        cards += f"""
        <div style="width:340px;border:1px solid #e0e0e0;border-radius:12px;overflow:hidden;
                    font-family:'Helvetica Neue',Arial,sans-serif;box-shadow:0 2px 12px rgba(0,0,0,.07);">
          <div style="position:relative;">
            <img src="{img_url}" onerror="this.onerror=null;this.parentNode.innerHTML='<div style=&quot;height:320px;background:#f3f3f3;display:flex;align-items:center;justify-content:center;color:#bbb;font-size:13px;&quot;>Image unavailable</div>';" style="width:100%;height:320px;object-fit:cover;display:block;" />
            <span style="position:absolute;top:12px;right:12px;background:#111;color:#fff;
                         font-size:10px;padding:3px 9px;border-radius:20px;letter-spacing:.5px;">NEW</span>
          </div>
          <div style="padding:18px;">
            <div style="font-size:11px;color:#888;letter-spacing:1px;text-transform:uppercase;">{p.category} · {p.id}</div>
            <div style="font-size:18px;font-weight:700;margin:4px 0 6px;">{p.name}</div>
            <div style="font-size:14px;color:#444;line-height:1.5;margin-bottom:12px;">{p.description}</div>
            <div style="margin-bottom:10px;">{colour_dots}</div>
            <div style="margin-bottom:12px;display:flex;gap:6px;flex-wrap:wrap;">{size_pills}</div>
            <div style="margin-bottom:14px;">{tags_html}</div>
            <div style="display:flex;justify-content:space-between;align-items:center;">
              <span style="font-size:22px;font-weight:800;">€{p.price:.2f}</span>
              <button style="background:#111;color:#fff;border:none;padding:10px 22px;
                             border-radius:6px;font-size:13px;cursor:pointer;">Add to Cart</button>
            </div>
          </div>
        </div>"""

    html = f"""
    <div style="background:#fafafa;padding:30px;border-radius:16px;">
      <div style="font-family:'Helvetica Neue',Arial,sans-serif;margin-bottom:24px;">
        <div style="font-size:11px;letter-spacing:3px;color:#888;text-transform:uppercase;">Silver Trust</div>
        <div style="font-size:28px;font-weight:900;letter-spacing:-0.5px;">New Arrivals</div>
        <div style="font-size:14px;color:#666;margin-top:4px;">Streetwear. Germany & Europe.</div>
      </div>
      <div style="display:flex;gap:24px;flex-wrap:wrap;">{cards}</div>
    </div>
    """
    display(HTML(html))

render_storefront(HERO_PRODUCTS)

## 4 · Recommendation Engine

**How it works:**
1. Each product is embedded once using `text-embedding-3-small`.
2. A simulated user session (viewed products) creates a **session embedding** (average of viewed item vectors).
3. Cosine similarity ranks the remaining catalog.
4. The call is traced in LangSmith under the `recommendation-engine` run name.

> This mirrors the *Session-based collaborative filter* flow described in the proposal.

In [ ]:
import numpy as np
from openai import OpenAI
from langsmith import traceable

openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])


def get_embedding(text: str) -> List[float]:
    """Single embedding call (no LangSmith trace needed — utility function)."""
    response = openai_client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=text,
    )
    return response.data[0].embedding


def cosine_similarity(a: List[float], b: List[float]) -> float:
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))


# ── Pre-compute embeddings for the whole catalog ─────────────────────────────
print("⏳ Embedding catalog... (one API call per product)")
for product in ALL_PRODUCTS:
    product.embedding = get_embedding(product.catalog_text())
    print(f"   ✓ {product.id} — {product.name}")

print(f"\n✅ {len(ALL_PRODUCTS)} products embedded")

In [ ]:
from langsmith import traceable
from typing import List, Dict


@traceable(name="recommendation-engine", run_type="retriever")
def recommend(
    viewed_product_ids: List[str],
    top_k: int = 3,
    exclude_viewed: bool = True,
) -> List[Dict]:
    """
    Given a list of product IDs the user has viewed, return the top-k
    most similar products from the rest of the catalog.

    Traced in LangSmith as 'recommendation-engine'.
    """
    # Build session embedding = mean of viewed product vectors
    viewed_vectors = [
        CATALOG_INDEX[pid].embedding
        for pid in viewed_product_ids
        if pid in CATALOG_INDEX and CATALOG_INDEX[pid].embedding is not None
    ]
    if not viewed_vectors:
        raise ValueError("No valid product IDs found in catalog.")

    session_vector = np.mean(viewed_vectors, axis=0).tolist()

    # Score all products not already viewed
    candidates = [
        p for p in ALL_PRODUCTS
        if (p.id not in viewed_product_ids if exclude_viewed else True)
        and p.embedding is not None
    ]

    scored = [
        {
            "product": p,
            "score": cosine_similarity(session_vector, p.embedding),
        }
        for p in candidates
    ]
    scored.sort(key=lambda x: x["score"], reverse=True)

    return [
        {
            "rank": i + 1,
            "id": r["product"].id,
            "name": r["product"].name,
            "category": r["product"].category,
            "price": r["product"].price,
            "similarity_score": round(r["score"], 4),
        }
        for i, r in enumerate(scored[:top_k])
    ]


print("✅ Recommendation engine ready")

In [ ]:
# ── Demo: user viewed the Köln Hoodie → what should we recommend? ────────────
viewed = ["ST-001"]   # Köln Oversized Hoodie

print(f"👤 User viewed: {', '.join(CATALOG_INDEX[pid].name for pid in viewed)}")
print("-" * 60)

recs = recommend(viewed_product_ids=viewed, top_k=3)

for r in recs:
    print(f"  #{r['rank']}  {r['id']}  {r['name']:<30}  €{r['price']:.2f}  "
          f"similarity={r['similarity_score']}")

print("\n📡 → Trace visible in LangSmith under project 'silver-trust-mvp'")

In [ ]:
# ── Demo: user viewed both hero products → cross-category recommendations ────
viewed2 = ["ST-001", "ST-002"]   # Hoodie + Cargo

print(f"👤 User viewed: {', '.join(CATALOG_INDEX[pid].name for pid in viewed2)}")
print("-" * 60)

recs2 = recommend(viewed_product_ids=viewed2, top_k=3)

for r in recs2:
    print(f"  #{r['rank']}  {r['id']}  {r['name']:<30}  €{r['price']:.2f}  "
          f"similarity={r['similarity_score']}")

print("\n📡 → Trace visible in LangSmith under project 'silver-trust-mvp'")

In [ ]:
# ── Render recommendation results as HTML cards ───────────────────────────────
def render_recommendations(viewed_ids: List[str], recs: List[Dict]):
    viewed_names = ", ".join(CATALOG_INDEX[pid].name for pid in viewed_ids)
    cards = ""
    for r in recs:
        p = CATALOG_INDEX[r["id"]]
        pct = int(r["similarity_score"] * 100)
        cards += f"""
        <div style="border:1px solid #e8e8e8;border-radius:10px;padding:16px;width:200px;
                    font-family:'Helvetica Neue',Arial,sans-serif;background:#fff;">
          <div style="font-size:10px;color:#aaa;text-transform:uppercase;letter-spacing:1px;">
            #{r['rank']} · {p.category}</div>
          <div style="font-size:15px;font-weight:700;margin:6px 0;">{p.name}</div>
          <div style="font-size:13px;color:#555;margin-bottom:10px;">{p.description[:70]}…</div>
          <div style="font-size:18px;font-weight:800;margin-bottom:8px;">€{p.price:.2f}</div>
          <div style="background:#f5f5f5;border-radius:6px;padding:4px 8px;font-size:11px;color:#444;">
            Match score: <strong>{pct}%</strong>
          </div>
        </div>"""

    html = f"""
    <div style="background:#f7f7f7;padding:24px;border-radius:14px;font-family:'Helvetica Neue',Arial,sans-serif;">
      <div style="font-size:11px;letter-spacing:2px;color:#999;text-transform:uppercase;">Recommendation Engine</div>
      <div style="font-size:18px;font-weight:700;margin:4px 0 6px;">Because you viewed: <em>{viewed_names}</em></div>
      <div style="font-size:13px;color:#777;margin-bottom:16px;">
        Embedding similarity · model: {EMBEDDING_MODEL} · traced in LangSmith</div>
      <div style="display:flex;gap:16px;flex-wrap:wrap;">{cards}</div>
    </div>"""
    display(HTML(html))


render_recommendations(viewed, recs)
print()
render_recommendations(viewed2, recs2)

## 5 · AI Styling Assistant

**How it works:**
1. The full catalog is injected into the system prompt as RAG context — no hallucinated products.
2. The assistant answers in Silver Trust's brand voice (direct, street-credible, never generic).
3. It always references real product IDs and prices.
4. Every turn is traced in LangSmith as `styling-assistant`.

> Mirrors Module 02 of the proposal: *Customer message → LLM + catalog context → Styled response + product links*

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langsmith import traceable

# ── Build the catalog RAG context (injected once into the system prompt) ──────
CATALOG_CONTEXT = "\n".join(p.catalog_text() for p in ALL_PRODUCTS)

SYSTEM_PROMPT = f"""You are the Silver Trust styling assistant — a sharp, knowledgeable streetwear advisor \
for a German e-commerce brand. Your tone is direct, confident, and street-credible. \
You speak like a trusted friend in the store, not a generic chatbot.

RULES:
- Only recommend products from the catalog below. Never invent items.
- Always include the product ID and price when recommending something.
- Keep responses concise (3–5 sentences max per turn).
- If you don't know something, say so honestly.
- You are AI-powered. If asked, confirm this clearly (EU AI Act compliance).

SILVER TRUST CATALOG:
{CATALOG_CONTEXT}
"""

llm = ChatOpenAI(
    model=CHAT_MODEL,
    temperature=0.7,
    openai_api_key=os.environ["OPENAI_API_KEY"],
)

print("✅ Styling assistant ready")
print(f"   Model: {CHAT_MODEL}")
print(f"   Catalog context: {len(CATALOG_CONTEXT)} characters injected into system prompt")

In [ ]:
from typing import List as LList

# Conversation history (stateful across turns in this session)
_conversation_history: LList = []


@traceable(name="styling-assistant", run_type="llm")
def ask_styling_assistant(user_message: str) -> str:
    """
    Send a message to the Silver Trust styling assistant.
    Maintains conversation history for the duration of the notebook session.
    Traced in LangSmith as 'styling-assistant'.
    """
    _conversation_history.append(HumanMessage(content=user_message))

    messages = [SystemMessage(content=SYSTEM_PROMPT)] + _conversation_history
    response = llm.invoke(messages)

    _conversation_history.append(AIMessage(content=response.content))
    return response.content


def reset_conversation():
    """Clear the conversation history to start a new session."""
    global _conversation_history
    _conversation_history = []
    print("🔄 Conversation reset")


def chat(message: str, render: bool = True):
    """Convenience wrapper: ask, print, optionally render as HTML."""
    reply = ask_styling_assistant(message)
    if render:
        _render_chat_bubble("You", message, is_user=True)
        _render_chat_bubble("Silver Trust Assistant", reply, is_user=False)
    else:
        print(f"You: {message}")
        print(f"\nAssistant: {reply}\n")
    return reply


def _render_chat_bubble(sender: str, text: str, is_user: bool):
    bg    = "#111" if is_user else "#fff"
    color = "#fff" if is_user else "#111"
    align = "flex-end" if is_user else "flex-start"
    border = "" if is_user else "border:1px solid #e0e0e0;"
    badge_bg = "#444" if is_user else "#f0f0f0"
    badge_color = "#fff" if is_user else "#333"
    html = f"""
    <div style="display:flex;justify-content:{align};margin:6px 0;font-family:'Helvetica Neue',Arial,sans-serif;">
      <div style="max-width:75%;">
        <div style="font-size:10px;color:#aaa;margin-bottom:3px;text-align:{'right' if is_user else 'left'}">
          <span style="background:{badge_bg};color:{badge_color};padding:2px 7px;
                       border-radius:10px;font-size:10px;">{sender}</span>
        </div>
        <div style="background:{bg};color:{color};{border}padding:12px 16px;
                    border-radius:{'14px 14px 4px 14px' if is_user else '14px 14px 14px 4px'};
                    font-size:14px;line-height:1.6;box-shadow:0 1px 4px rgba(0,0,0,.08);">
          {text.replace(chr(10), '<br>')}
        </div>
      </div>
    </div>"""
    display(HTML(html))


print("✅ Chat interface ready  —  use chat('your message') to talk to the assistant")

In [ ]:
# ── Demo conversation — Turn 1 ──────────────────────────────────────────────
display(HTML("""
<div style="background:#fafafa;padding:16px 20px;border-radius:12px;margin-bottom:8px;
            font-family:'Helvetica Neue',Arial,sans-serif;border:1px solid #eee;">
  <span style="font-size:11px;letter-spacing:2px;color:#999;text-transform:uppercase;">AI Styling Assistant</span>
  <span style="margin-left:10px;font-size:11px;background:#111;color:#fff;padding:2px 8px;
               border-radius:10px;">AI-powered · LangSmith traced</span>
  <div style="font-size:17px;font-weight:700;margin-top:4px;">Silver Trust — In-store advisor, online</div>
</div>
"""))

chat("What goes with the Köln hoodie?")

In [ ]:
# ── Turn 2 — size advice ────────────────────────────────────────────────────
chat("I'm 180cm, 75kg. Should I go M or L in the Köln hoodie?")

In [ ]:
# ── Turn 3 — complete outfit request ────────────────────────────────────────
chat("Build me a full outfit under €250 — heading to a streetwear market.")

In [ ]:
# ── Turn 4 — EU AI Act compliance check ─────────────────────────────────────
chat("Are you a real person or an AI?")

## 6 · LangSmith Trace Summary

All calls made in this notebook are visible in your LangSmith project.

In [ ]:
from langsmith import Client as LangSmithClient

ls_client = LangSmithClient(
    api_key=os.environ["LANGSMITH_API_KEY"],
    api_url=os.environ["LANGSMITH_ENDPOINT"],
)

project_name = os.environ["LANGSMITH_PROJECT"]

try:
    runs = list(ls_client.list_runs(
        project_name=project_name,
        limit=20,
    ))

    if not runs:
        print("No runs found yet — traces may still be flushing (wait a few seconds and retry).")
    else:
        rows = []
        for r in runs:
            latency = ""
            if r.end_time and r.start_time:
                latency = f"{(r.end_time - r.start_time).total_seconds():.2f}s"
            tokens = ""
            if r.total_tokens:
                tokens = str(r.total_tokens)
            rows.append({
                "run_name": r.name,
                "run_type": r.run_type,
                "status":   r.status,
                "latency":  latency,
                "tokens":   tokens,
                "started":  r.start_time.strftime("%H:%M:%S") if r.start_time else "",
            })

        # Render as HTML table
        header_cells = "".join(
            f'<th style="padding:8px 14px;text-align:left;font-size:11px;letter-spacing:1px;"\
               >{k.upper()}</th>'
            for k in rows[0].keys()
        )
        body_rows = ""
        for i, row in enumerate(rows):
            bg = "#fafafa" if i % 2 == 0 else "#fff"
            status_color = "#22c55e" if row["status"] == "success" else "#ef4444"
            cells = ""
            for k, v in row.items():
                if k == "status":
                    cells += f'<td style="padding:7px 14px;font-size:12px;color:{status_color};"><strong>{v}</strong></td>'
                else:
                    cells += f'<td style="padding:7px 14px;font-size:12px;">{v}</td>'
            body_rows += f'<tr style="background:{bg};">{cells}</tr>'

        table_html = f"""
        <div style="font-family:'Helvetica Neue',Arial,sans-serif;margin-top:10px;">
          <div style="font-size:11px;letter-spacing:2px;color:#999;text-transform:uppercase;">LangSmith</div>
          <div style="font-size:17px;font-weight:700;margin:4px 0 14px;">Trace log — project: {project_name}</div>
          <table style="border-collapse:collapse;width:100%;border:1px solid #e0e0e0;border-radius:8px;overflow:hidden;">
            <thead style="background:#111;color:#fff;">{header_cells}</thead>
            <tbody>{body_rows}</tbody>
          </table>
          <div style="font-size:12px;color:#888;margin-top:8px;">
            Full traces (inputs, outputs, token costs, latency breakdowns) → 
            <a href="https://smith.langchain.com" target="_blank">smith.langchain.com</a>
          </div>
        </div>"""
        display(HTML(table_html))

except Exception as e:
    print(f"Could not fetch runs from LangSmith: {e}")
    print("Traces are still being sent — check smith.langchain.com directly.")

## 7 · Interactive Free-Chat

Use this cell to continue the conversation with the styling assistant freely.
Every message is traced in LangSmith.

In [ ]:
# Reset conversation history for a clean session
reset_conversation()

# Type your own question here ↓
chat("Do you have anything that would work for cold weather but still looks clean?")

In [ ]:
# Continue the conversation — just re-run this cell with a new message
chat("What about accessories to finish the look?")

---

## Architecture Summary

| Component | Technology | LangSmith run name |
|-----------|-----------|--------------------|
| Storefront | HTML/CSS rendered in notebook | — |
| Recommendation engine | `text-embedding-3-small` + cosine similarity | `recommendation-engine` |
| AI Styling Assistant | `gpt-4o-mini` + RAG (full catalog in prompt) | `styling-assistant` |
| Observability | LangSmith tracing v2 | project: `silver-trust-mvp` |

**Next steps toward production (Phase 2):**
- Move embeddings to a vector database (pgvector or Pinecone) for sub-80ms retrieval at scale
- Add user session IDs to LangSmith metadata for per-customer trace filtering
- Wrap Flask/FastAPI around `recommend()` and `ask_styling_assistant()` to expose as REST endpoints
- Set LangSmith budget alerts for token cost monitoring (as specified in proposal §7)